In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import scipy
from IPython.display import VimeoVideo
from pymongo import MongoClient
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.power import GofChisquarePower
from teaching_tools.ab_test.experiment import Experiment
from teaching_tools.ab_test.reset import Reset

# Reset database
r = Reset("192.59.115.2")
r.reset_database()


In [ ]:
VimeoVideo("742459144", h="0f1aa2db83", width=600)


In [ ]:
host = "192.59.115.2"


In [ ]:
client = MongoClient(host=host, port=27017)
ds_app = client["wqu-abtest"]["ds-applicants"]
print("client:", type(client))
print("ds_app:", type(ds_app))


In [ ]:
VimeoVideo("734517993", h="624e1cd2ea", width=600)


In [ ]:
VimeoVideo("734517709", h="907b2d3102", width=600)


In [ ]:
chi_square_power = GofChisquarePower()
group_size = math.ceil(chi_square_power.solve_power(effect_size=0.2, alpha=.05, power=0.8))

print("Group size:", group_size)
print("Total # of applicants needed:", group_size * 2)


In [ ]:
VimeoVideo("734517244", h="44460ba891", width=600)


In [ ]:
n_observations = np.arange(0, group_size * 2 + 1)
effect_sizes = np.array([0.2, 0.5, 0.8])

chi_square_power.plot_power(
    dep_var="nobs",
    nobs=n_observations,
    effect_size=effect_sizes,
    alpha=0.05,
    n_bins=2
);


In [ ]:
VimeoVideo("734516984", h="f8c2ae9e0e", width=600)


In [ ]:
result = ds_app.aggregate(
    [
        {"$match": {"admissionsQuiz": "incomplete"}},
        {
            "$group": {
                "_id": {
                    "$dateTrunc": {
                        "date": "$createdAt",
                        "unit": "day"
                    }
                },
                "count": {"$sum": 1}
            }
        }
    ]
)

print("result type:", type(result))


In [ ]:
VimeoVideo("734516829", h="9c7014eb8d", width=600)


In [ ]:
no_quiz = (
    pd.DataFrame(result)
    .rename({"_id": "date", "count": "new_users"}, axis=1)
    .set_index("date")
    .sort_index()
    .squeeze()
)

print("no_quiz type:", type(no_quiz))
print("no_quiz shape:", no_quiz.shape)
no_quiz.head()


In [ ]:
VimeoVideo("734516524", h="c1e506e702", width=600)


In [ ]:
# Create histogram of `no_quiz`
no_quiz.hist()
# Add axis labels and title
plt.xlabel("No-Quiz Applicants")
plt.ylabel("Frequency [count]")
plt.title("Distribution of Daily No-Quiz Applicants")


In [ ]:
VimeoVideo("734516130", h="a93fabac0f", width=600)


In [ ]:
mean = no_quiz.describe()["mean"]
std = no_quiz.describe()["std"]
print("no_quiz mean:", mean)
print("no_quiz std:", std)


In [ ]:
VimeoVideo("742459088", h="1962b016f9", width=600)


In [ ]:
days = 10
sum_mean = mean * days
sum_std = std * np.sqrt(days)
print("Mean of sum:", sum_mean)
print("Std of sum:", sum_std)


In [ ]:
VimeoVideo("742459015", h="33ad7b37ca", width=600)


In [ ]:
prob_400_or_fewer = scipy.stats.norm.cdf(
    group_size * 2,
    loc=sum_mean,
    scale=sum_std
)

prob_400_or_greater = 1 - prob_400_or_fewer

print(
    f"Probability of getting 400+ no_quiz in {days} days:",
    round(prob_400_or_greater, 3),
)


In [ ]:
VimeoVideo("734515713", h="7702f5163d", width=600)


In [ ]:
exp = Experiment(repo=client, db="wqu-abtest", collection="ds-applicants")
exp.reset_experiment()
result = exp.run_experiment(days=days)
print("result type:", type(result))
result


In [ ]:
VimeoVideo("734515601", h="759340caf1", width=600)


In [ ]:
result = ds_app.find({"inExperiment": True})
print("results type:", type(result))


In [ ]:
VimeoVideo("734515308", h="8308ce4a22", width=600)


In [ ]:
df = pd.DataFrame(result)

print("df type:", type(df))
print("df shape:", df.shape)
df.head()


In [ ]:
VimeoVideo("734514187", h="9063c1eccf", width=600)


In [ ]:
data = pd.crosstab(
    index=df["group"],
    columns=df["admissionsQuiz"],
    normalize=False
)

print("data type:", type(data))
print("data shape:", data.shape)
data


In [ ]:
VimeoVideo("734513651", h="cc012589ac", width=600)


In [ ]:
def build_contingency_bar():
    # Create side-by-side bar chart
    fig = px.bar(
        data_frame=data,
        barmode="group",
        title="Admissions Quiz Completion by Group"
    )

    # Set axis labels
    fig.update_layout(
        xaxis_title="Group",
        yaxis_title="Frequency [count]",
        legend={"title": "Admissions Quiz"}
    )

    return fig
    
con_bar = build_contingency_bar()
print("con_bar type:", type(con_bar))
con_bar.show()


In [ ]:
VimeoVideo("734512752", h="92e79c3f89", width=600)


In [ ]:
contingency_table = Table2x2(data.values)

print("contingency_table type:", type(contingency_table))
contingency_table.table_orig


In [ ]:
VimeoVideo("734512565", h="4e29a856e1", width=600)


In [ ]:
# Calculate fitted values
contingency_table.fittedvalues


In [ ]:
VimeoVideo("734512366", h="70d4db3edd", width=600)


In [ ]:
# Calculate independent joint probabilities
contingency_table.independence_probabilities.round(3)


In [ ]:
VimeoVideo("742458959", h="e8da1aeecf", width=600)


In [ ]:
chi_square_test = contingency_table.test_nominal_association()

print("chi_square_test type:", type(chi_square_test))
print(chi_square_test)


In [ ]:
VimeoVideo("734512125", h="8dbc500ec2", width=600)


In [ ]:
odds_ratio = contingency_table.oddsratio.round(1)
print("Odds ratio:", odds_ratio)


In [ ]:
VimeoVideo("748065153", h="47f74a0df8", width=600)


In [ ]:
summary = contingency_table.summary()
print("summary type:", type(summary))
summary
